In [1]:
import pandas as pd

In [2]:
movie_des = pd.read_csv("../data/enriched_movie_descriptions.csv")

In [3]:
movie_original = pd.read_csv("../data/movie.csv")
movie_rating= pd.read_csv("../data/rating.csv")

In [4]:
movie_rating.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [5]:
movie_des.head()

,movieId,tmdbId,title,description,poster
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",/8MprEuTY3EwkF9nBBPCUyjRjvs5.jpg
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg


In [6]:
movie_des.count()

movieId        25927
tmdbId         25927
title          25927
description    25912
poster         25777
dtype: int64

In [7]:
movie_original.count()

movieId    27278
title      27278
genres     27278
dtype: int64

In [8]:
import pandas as pd

# Step 1: Remove rows in 'movie_des' where the 'description' is missing
# This leaves us with the 26,619 movies that actually have a description.
movie_des_filtered = movie_des.dropna(subset=['description'])

# Step 2: Combine 'movie_original' with the filtered descriptions
# We only take 'movieId' and 'description' from the filtered dataset 
# to avoid getting duplicate columns for 'title' (like 'title_x' and 'title_y').
process_movie = pd.merge(
    movie_original, 
    movie_des_filtered[['movieId', 'description', 'poster']], 
    on='movieId',
    how='inner' 
)

# Step 3: Rearrange columns to match your exact requested format (optional but clean)
process_movie = process_movie[['movieId', 'title', 'genres', 'description', 'poster']]

# View the results
print(process_movie.info())

<class 'pandas.DataFrame'>
RangeIndex: 25912 entries, 0 to 25911
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   movieId      25912 non-null  int64
 1   title        25912 non-null  str  
 2   genres       25912 non-null  str  
 3   description  25912 non-null  str  
 4   poster       25765 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1012.3 KB
None


In [9]:
process_movie.head()

,movieId,title,genres,description,poster
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"Led by Woody, Andy's toys live happily in his ...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg
1,2,Jumanji (1995),Adventure|Children|Fantasy,When siblings Judy and Peter discover an encha...,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg
2,3,Grumpier Old Men (1995),Comedy|Romance,A family wedding reignites the ancient feud be...,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"Cheated on, mistreated and stepped on, the wom...",/8MprEuTY3EwkF9nBBPCUyjRjvs5.jpg
4,5,Father of the Bride Part II (1995),Comedy,Just when George Banks has recovered from his ...,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg


In [10]:
process_movie['year'] = process_movie['title'].str.extract(r'\((\d{4})\)')

In [11]:
# Remove the space and the year in parentheses from the end of the title
# r'\s*\(\d{4}\)$' looks for: a space, then (four digits), at the end of the text
process_movie['title'] = process_movie['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)

In [12]:
# Replace the pipe '|' with a comma and a space ', '
process_movie['genres'] = process_movie['genres'].str.replace('|', ',', regex=False)

In [13]:
process_movie.head()

,movieId,title,genres,description,poster,year
0,1,Toy Story,"Adventure,Animation,Children,Comedy,Fantasy","Led by Woody, Andy's toys live happily in his ...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,1995
1,2,Jumanji,"Adventure,Children,Fantasy",When siblings Judy and Peter discover an encha...,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,1995
2,3,Grumpier Old Men,"Comedy,Romance",A family wedding reignites the ancient feud be...,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,1995
3,4,Waiting to Exhale,"Comedy,Drama,Romance","Cheated on, mistreated and stepped on, the wom...",/8MprEuTY3EwkF9nBBPCUyjRjvs5.jpg,1995
4,5,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,1995


In [14]:
# Step 4: Filter movie_rating to keep only IDs that exist in process_movie
movie_rating = movie_rating[movie_rating['movieId'].isin(process_movie['movieId'])]

# Optional: Check how many rows are left
print(f"Ratings remaining: {len(movie_rating)}")
print(movie_rating.head())

Ratings remaining: 19937428
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [15]:
# Save the DataFrame to a CSV file in your current working directory
process_movie.to_csv('../data/process_movie.csv', index=False)

In [16]:
movie_rating.to_csv('../data/process_movie_rating.csv', index=False)